# KS paper plots

In [ ]:
import h5py
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle, ConnectionPatch

cmap_heat = plt.colormaps["RdBu"].reversed()
cmap_heatR = plt.colormaps["Reds"]
cmap_heatB = plt.colormaps["Blues"].reversed()

Lfact = 7 # domain length factor
Ldom = Lfact * np.pi # domain length
dt = 0.25

Ntrue = 575 # time snapshots
Nq = 64 # delays
L = 256

N = Ntrue - Nq + 1 # temporal space dimension
M = 64 # spatial space dimension
NM = N * M # product space dimension

nvec = np.arange(N)
xvec = Ldom * np.arange(-int(M/2), int(M/2))/M

print(N, M, NM, L)

## State heat maps

In [ ]:
M2 = 128
xvec2 = 30*np.pi * np.arange(-int(M2/2), int(M2/2))/M2

fname = f"data/ks_true_{Ntrue}_{M}_7_u32.h5"
with h5py.File(fname, "r") as f:
    uvec = f["u"][:].reshape((Ntrue, M), order="F")[:N, :]
    tvec = f["t"][:N]
tvec -= tvec[0]

fname = f"data/ks_true_{Ntrue}_{M2}_30_u32.h5"
with h5py.File(fname, "r") as f:
    uvec2 = f["u"][:].reshape((Ntrue, M2), order="F")[:N, :]

In [ ]:
def _detail_box(ax, ylo, yhi, xvec, overshoot=0.03):
    xlo, xhi = xvec.min(), xvec.max()
    pad = (xhi - xlo) * overshoot
    xlo, xhi = xlo - pad, xhi + pad
    rect = Rectangle((xlo, ylo), xhi - xlo, yhi - ylo,
        fill=False, edgecolor="k", ls="--", lw=1.0, clip_on=False)
    rect.set_in_layout(False)
    ax.add_artist(rect)

def _detail_arrow(axA, xyA, axB, xyB, rad):
    con = ConnectionPatch(xyA=xyA, coordsA=axA.transData,
        xyB=xyB, coordsB=axB.transAxes,
        arrowstyle="-|>", connectionstyle=f"arc3,rad={rad}",
        color="k", lw=1.0, mutation_scale=12)
    con.set_in_layout(False)
    con.set_clip_on(False)
    fig.add_artist(con)

In [ ]:
Nsub = 4000

fig = plt.figure(layout="tight")
gs = fig.add_gridspec(2, 4)

ax0 = fig.add_subplot(gs[:, 0])
ax1t = fig.add_subplot(gs[0, 1])
ax1b = fig.add_subplot(gs[1, 1])

ax2t = fig.add_subplot(gs[0, 2])
ax2b = fig.add_subplot(gs[1, 2])
ax3 = fig.add_subplot(gs[:, 3])

pcm0 = ax0.pcolormesh(xvec, tvec, uvec, cmap=cmap_heat)
ax0.tick_params(axis="x", length=0)
ax0.tick_params(axis="y", length=0)
ax0.ticklabel_format(axis="y", style="sci", scilimits=(0,0))
ax0.get_yaxis().get_offset_text().set_position((-0.4, 0.0))
ax0.set_xlabel("Space")
ax0.set_ylabel("Time")
ax0.set_title(r"$L=7\pi$", pad=35)
fig.colorbar(pcm0, ax=ax0, location="top", pad=0.01, fraction=0.07)
_detail_box(ax0, tvec[0], tvec[Nsub], xvec)
_detail_box(ax0, tvec[-Nsub], tvec[-1], xvec)

pcm3 = ax3.pcolormesh(xvec2, tvec, uvec2, cmap=cmap_heat)
ax3.tick_params(axis="x", length=0)
ax3.tick_params(axis="y", length=0)
ax3.set_yticklabels([])
ax3.set_title(r"$L=30\pi$", pad=35)
fig.colorbar(pcm3, ax=ax3, location="top", pad=0.01, fraction=0.07)
_detail_box(ax3, tvec[0], tvec[Nsub], xvec2)
_detail_box(ax3, tvec[-Nsub], tvec[-1], xvec2)

pcm1t = ax1t.pcolormesh(xvec, tvec[-Nsub:], uvec[-Nsub:, :], cmap=cmap_heat,
                        vmin=uvec.min(), vmax=uvec.max())
ax1t.tick_params(axis="x", length=0)
ax1t.tick_params(axis="y", length=0)
ax1t.set_xticklabels([])
ax1t.set_yticklabels([])

t1t = 0.5 * (tvec[-Nsub] + tvec[-1])
x1t = xvec.max() + 0.02 * (xvec.max() - xvec.min())
_detail_arrow(ax0, (x1t, t1t), ax1t, (-0.01, 0.65), rad=0)

pcm1b = ax1b.pcolormesh(xvec, tvec[:Nsub], uvec[:Nsub, :], cmap=cmap_heat,
                        vmin=uvec.min(), vmax=uvec.max())
ax1b.tick_params(axis="x", length=0)
ax1b.tick_params(axis="y", length=0)
ax1b.set_xticklabels([])
ax1b.set_yticklabels([])

t1b = 0.5 * (tvec[0] + tvec[Nsub])
x1b = xvec.max() + 0.02 * (xvec.max() - xvec.min())
_detail_arrow(ax0, (x1b, t1b), ax1b, (-0.01, 0.25), rad=0)

pcm2t = ax2t.pcolormesh(xvec2, tvec[-Nsub:], uvec2[-Nsub:, :], cmap=cmap_heat,
                        vmin=uvec2.min(), vmax=uvec2.max())
ax2t.tick_params(axis="x", length=0)
ax2t.tick_params(axis="y", length=0)
ax2t.set_xticklabels([])
ax2t.set_yticklabels([])

t2t = 0.5 * (tvec[-Nsub] + tvec[-1])
x2t = xvec2.min() - 0.02 * (xvec2.max() - xvec2.min())
_detail_arrow(ax3, (x2t, t2t), ax2t, (1.02, 0.65), rad=0)

pcm2b = ax2b.pcolormesh(xvec2, tvec[:Nsub], uvec2[:Nsub, :], cmap=cmap_heat,
                        vmin=uvec2.min(), vmax=uvec2.max())
ax2b.tick_params(axis="x", length=0)
ax2b.tick_params(axis="y", length=0)
ax2b.set_xticklabels([])
ax2b.set_yticklabels([])

t2b = 0.5 * (tvec[0] + tvec[Nsub])
x2b = xvec2.min() - 0.02 * (xvec2.max() - xvec2.min())
_detail_arrow(ax3, (x2b, t2b), ax2b, (1.02, 0.25), rad=0)

#fig.savefig("data/figs/state.png")

## Heat map & eigenvalues comparison

In [ ]:
fname = f"data/ks_true_{Ntrue}_{M}_{Lfact}_u32.h5"
with h5py.File(fname, "r") as f:
    uvec = f["u"][:].reshape((Ntrue, M), order="F")
    tvec = f["t"][:]
tvec -= tvec[0]

fname = f"data/evd_ref_{N}_{M}_{Nq}_{Lfact}.h5"
with h5py.File(fname, "r") as f:
    eigref = f["eigvals"][:]

fname = f"data/evd_{N}_{M}_{Nq}_{L}_{Lfact}.h5"
with h5py.File(fname, "r") as f:
    eig = f["lam"][:]

eig = eig[::-1]
print(eigref.shape, eig.shape)

In [ ]:
fig = plt.figure(layout="tight")
gs = GridSpec(2, 2, figure=fig)

ax0 = fig.add_subplot(gs[:, 0])
pcm0 = ax0.pcolormesh(xvec, tvec, uvec, cmap=cmap_heat)
ax0.tick_params(axis="x", length=0)
ax0.tick_params(axis="y", length=0)
ax0.set_xlabel("Space")
ax0.set_ylabel("Time")
ax0.set_title(r"$u$", pad=35)
fig.colorbar(pcm0, ax=ax0, location="top", pad=0.01, fraction=0.07)

ax1t = fig.add_subplot(gs[0, 1])
ax1t.plot(eigref, label="True")
ax1t.plot(eig, ls="--", label="Approx.")
ax1t.set_xscale("log")
ax1t.set_yscale("log")
ax1t.set_xlim([1, 2e3])
ax1t.tick_params(axis="both", which="both", direction="in")
ax1t.tick_params(right=True, labelright=True, labelleft=False)
ax1t.legend(loc="lower left")
ax1t.set_title("Eigenvalues", pad=10)
ax1t.grid(lw=0.5, alpha=0.5)

ax1b = fig.add_subplot(gs[1, 1])
ax1b.plot(eigref[:20], ".", label="True")
ax1b.plot(eig[:20], "x", label="Approx.")
ax1b.set_yscale("log")
ax1b.set_xlabel("Index")
ax1b.tick_params(axis="both", which="both", direction="in")
ax1b.tick_params(right=True, labelright=True, labelleft=False)
ax1b.legend(loc="upper right")
ax1b.grid(which="major", lw=0.5, alpha=0.5)

#fig.savefig("data/figs/compare1.png")

## Eigenfunctions comparison

In [ ]:
fname = f"data/evd_ref_{N}_{M}_{Nq}_{Lfact}.h5"
with h5py.File(fname, "r") as f:
    Phiref = f["eigvecs"][:].reshape((NM, NM), order="F")

fname = f"data/evd_{N}_{M}_{Nq}_{L}_{Lfact}.h5"
with h5py.File(fname, "r") as f:
    Phi = f["U"][:].reshape((NM, L), order="C")

tvec2 = tvec[:N]

In [ ]:
f0ref = -Phiref[:, 1].reshape((N, M), order="C")
f0 = Phi[:, -1-1].reshape((N, M), order="F")

f1ref = -Phiref[:, 2].reshape((N, M), order="C")
f1 = Phi[:, -1-2].reshape((N, M), order="F")

fig, ((ax0, ax1, ax2), (ax3, ax4, ax5)) = plt.subplots(2, 3, layout="tight", figsize=[6.4, 6.4])

pcm0 = ax0.pcolormesh(xvec, tvec2, f0ref, cmap=cmap_heat)
ax0.tick_params(axis="x", length=0)
ax0.tick_params(axis="y", length=0)
ax0.set_ylabel("Time")
ax0.set_title("True", pad=35)
fig.colorbar(pcm0, ax=ax0, location="top", pad=0.01, fraction=0.07)

pcm1 = ax1.pcolormesh(xvec, tvec2, f0, cmap=cmap_heat)
ax1.tick_params(axis="x", length=0)
ax1.tick_params(axis="y", length=0)
ax1.set_yticklabels([])
ax1.set_title("Approximate", pad=35)
fig.colorbar(pcm1, ax=ax1, location="top", pad=0.01, fraction=0.07)

pcm2 = ax2.pcolormesh(xvec, tvec2, np.abs(f0-f0ref), cmap=cmap_heat)
ax2.tick_params(axis="x", length=0)
ax2.tick_params(axis="y", length=0)
ax2.set_yticklabels([])
ax2.set_title("Error", pad=35)
fig.colorbar(pcm2, ax=ax2, location="top", pad=0.01, fraction=0.07)

pcm3 = ax3.pcolormesh(xvec, tvec2, f1ref, cmap=cmap_heat)
ax3.tick_params(axis="x", length=0)
ax3.tick_params(axis="y", length=0)
ax3.set_xlabel("Space")
ax3.set_ylabel("Time")
fig.colorbar(pcm3, ax=ax3, location="top", pad=0.01, fraction=0.07)

pcm4 = ax4.pcolormesh(xvec, tvec2, f1, cmap=cmap_heat)
ax4.tick_params(axis="x", length=0)
ax4.tick_params(axis="y", length=0)
ax4.set_yticklabels([])
fig.colorbar(pcm4, ax=ax4, location="top", pad=0.01, fraction=0.07)

pcm5 = ax5.pcolormesh(xvec, tvec2, np.abs(f1-f1ref), cmap=cmap_heat)
ax5.tick_params(axis="x", length=0)
ax5.tick_params(axis="y", length=0)
ax5.set_yticklabels([])
cbar = fig.colorbar(pcm5, ax=ax5, location="top", pad=0.01, fraction=0.07)
cbar.formatter.set_scientific(True)
cbar.formatter.set_powerlimits((0, 0))
cbar.update_ticks()

#fig.savefig("data/figs/compare2.png")

## Eigenfunction plots

In [ ]:
fname = f"data/evd_{N}_{M}_{Nq}_{L}_{Lfact}.h5"
with h5py.File(fname, "r") as f:
    Phi = f["U"][:].reshape((NM, L), order="C")

fname = f"data/ks_true_{Ntrue}_{M}_{Lfact}_u32.h5"
with h5py.File(fname, "r") as f:
    uvec = f["u"][:].reshape((Ntrue, M), order="F")[:N, :]
    tvec = f["t"][:N]
tvec -= tvec[0]

In [ ]:
# 1, 2, 7, 9
# 1, 2, 3, 5
i0 = 1 # 11, 19, 21, 23, 25, 27
i1 = 2 # 2, 12
i2 = 7
i3 = 9

f0 = Phi[:, -1-i0].reshape((N, M), order="F")
f1 = Phi[:, -1-i1].reshape((N, M), order="F")
f2 = Phi[:, -1-i2].reshape((N, M), order="F")
f3 = Phi[:, -1-i3].reshape((N, M), order="F")

fig, (ax0, ax1, ax2, ax3, ax4) = plt.subplots(1, 5, layout="tight")

pcm0 = ax0.pcolormesh(xvec, tvec, uvec, cmap=cmap_heat)
ax0.tick_params(axis="x", length=0)
ax0.tick_params(axis="y", length=0)
ax0.ticklabel_format(axis="y", style="sci", scilimits=(0,0))
ax0.get_yaxis().get_offset_text().set_position((-0.4, 0.0))
ax0.set_xlabel("Space")
ax0.set_ylabel("Time")
ax0.set_title(r"$u$", pad=40)
fig.colorbar(pcm0, ax=ax0, location="top", pad=0.01, fraction=0.07)

pcm1 = ax1.pcolormesh(xvec, tvec, f0, cmap=cmap_heat)
ax1.tick_params(axis="x", length=0)
ax1.tick_params(axis="y", length=0)
ax1.set_yticklabels([])
ax1.set_title(rf"$\phi_{ {i0} }$", pad=40)
cbar1 = fig.colorbar(pcm1, ax=ax1, location="top", pad=0.01, fraction=0.07)
cbar1.formatter.set_scientific(True)
cbar1.formatter.set_powerlimits((0, 0))
cbar1.update_ticks()

pcm2 = ax2.pcolormesh(xvec, tvec, f1, cmap=cmap_heat)
ax2.tick_params(axis="x", length=0)
ax2.tick_params(axis="y", length=0)
ax2.set_yticklabels([])
ax2.set_title(rf"$\phi_{ {i1} }$", pad=40)
cbar2 = fig.colorbar(pcm2, ax=ax2, location="top", pad=0.01, fraction=0.07)
cbar2.formatter.set_scientific(True)
cbar2.formatter.set_powerlimits((0, 0))
cbar2.update_ticks()

pcm3 = ax3.pcolormesh(xvec, tvec, f2, cmap=cmap_heat)
ax3.tick_params(axis="x", length=0)
ax3.tick_params(axis="y", length=0)
ax3.set_yticklabels([])
ax3.set_title(rf"$\phi_{ {i2} }$", pad=40)
cbar3 = fig.colorbar(pcm3, ax=ax3, location="top", pad=0.01, fraction=0.07)
cbar3.formatter.set_scientific(True)
cbar3.formatter.set_powerlimits((0, 0))
cbar3.update_ticks()

pcm4 = ax4.pcolormesh(xvec, tvec, f3, cmap=cmap_heat)
ax4.tick_params(axis="x", length=0)
ax4.tick_params(axis="y", length=0)
ax4.set_yticklabels([])
ax4.set_title(rf"$\phi_{ {i3} }$", pad=40)
cbar4 = fig.colorbar(pcm4, ax=ax4, location="top", pad=0.01, fraction=0.07)
cbar4.formatter.set_scientific(True)
cbar4.formatter.set_powerlimits((0, 0))
cbar4.update_ticks()

#fig.savefig("data/figs/eigfun1a.png")

In [ ]:
Nsub = 6000

fig, (ax0, ax1, ax2, ax3, ax4) = plt.subplots(1, 5, layout="tight")

pcm0 = ax0.pcolormesh(xvec, tvec[:Nsub], uvec[:Nsub, :], cmap=cmap_heat)
ax0.tick_params(axis="x", length=0)
ax0.tick_params(axis="y", length=0)
ax0.ticklabel_format(axis="y", style="sci", scilimits=(0,0))
ax0.get_yaxis().get_offset_text().set_position((-0.4, 0.0))
ax0.set_xlabel("Space")
ax0.set_ylabel("Time")
ax0.set_title(r"$u$", pad=40)
fig.colorbar(pcm0, ax=ax0, location="top", pad=0.01, fraction=0.07)

pcm1 = ax1.pcolormesh(xvec, tvec[:Nsub], f0[:Nsub, :], cmap=cmap_heat)
ax1.tick_params(axis="x", length=0)
ax1.tick_params(axis="y", length=0)
ax1.set_yticklabels([])
ax1.set_title(rf"$\phi_{ {i0} }$", pad=40)
cbar1 = fig.colorbar(pcm1, ax=ax1, location="top", pad=0.01, fraction=0.07)
cbar1.formatter.set_scientific(True)
cbar1.formatter.set_powerlimits((0, 0))
cbar1.update_ticks()

pcm2 = ax2.pcolormesh(xvec, tvec[:Nsub], f1[:Nsub, :], cmap=cmap_heat)
ax2.tick_params(axis="x", length=0)
ax2.tick_params(axis="y", length=0)
ax2.set_yticklabels([])
ax2.set_title(rf"$\phi_{ {i1} }$", pad=40)
cbar2 = fig.colorbar(pcm2, ax=ax2, location="top", pad=0.01, fraction=0.07)
cbar2.formatter.set_scientific(True)
cbar2.formatter.set_powerlimits((0, 0))
cbar2.update_ticks()

pcm3 = ax3.pcolormesh(xvec, tvec[:Nsub], f2[:Nsub, :], cmap=cmap_heat)
ax3.tick_params(axis="x", length=0)
ax3.tick_params(axis="y", length=0)
ax3.set_yticklabels([])
ax3.set_title(rf"$\phi_{ {i2} }$", pad=40)
cbar3 = fig.colorbar(pcm3, ax=ax3, location="top", pad=0.01, fraction=0.07)
cbar3.formatter.set_scientific(True)
cbar3.formatter.set_powerlimits((0, 0))
cbar3.update_ticks()

pcm4 = ax4.pcolormesh(xvec, tvec[:Nsub], f3[:Nsub, :], cmap=cmap_heat)
ax4.tick_params(axis="x", length=0)
ax4.tick_params(axis="y", length=0)
ax4.set_yticklabels([])
ax4.set_title(rf"$\phi_{ {i3} }$", pad=40)
cbar4 = fig.colorbar(pcm4, ax=ax4, location="top", pad=0.01, fraction=0.07)
cbar4.formatter.set_scientific(True)
cbar4.formatter.set_powerlimits((0, 0))
cbar4.update_ticks()

#fig.savefig("data/figs/eigfun1b.png")

## Scaling data

In [ ]:
filename = "data/scaling/scaling.txt"

rvec, tarpc, tevd, errarpc = np.loadtxt(filename, skiprows=3, unpack=True)
rvec = rvec.astype(int)

tarpc = tarpc / tarpc[0]
tevd = tevd / tevd[0]

fig, (ax0, ax1) = plt.subplots(1, 2, layout="tight", figsize=(9.6, 4.8))

ax0.loglog(rvec, tarpc, marker="x", ms=6.0, lw=1, label="ARPC-LM")
ax0.loglog(rvec, tevd, marker=".", ms=8.0, lw=1, label="EVD")
ax0.loglog(rvec, tarpc[-1] * (rvec / rvec[-1])**2, "--", lw=1, label=r"$r^2$")
ax0.set_xscale("log", base=2)
ax0.set_xticks(rvec)
ax0.xaxis.set_major_formatter(ScalarFormatter())
ax0.tick_params(axis="x", rotation=45)
ax0.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax0.set_xlabel(r"Rank $r$")
ax0.set_ylabel("Normalized time")
ax0.grid(lw=0.5, alpha=0.5)
ax0.legend()

ax1.loglog(rvec, errarpc, marker=".", ms=8.0, lw=1, label="ARPC-LM")
ax1.set_xscale("log", base=2)
ax1.xaxis.set_major_formatter(ScalarFormatter())
ax1.tick_params(axis="x", rotation=45)
ax1.tick_params(axis="both", which="both", direction="in", top=True, right=True)
ax1.tick_params(axis="y", which="both", right=True, labelright=True, labelleft=False)
ax1.set_xlabel(r"Rank $r$")
ax1.set_ylabel("Relative error", labelpad=10)
ax1.yaxis.set_label_position("right")
ax1.grid(lw=0.5, alpha=0.5)
ax1.legend()

#fig.savefig("data/figs/scaling.png")